In [1]:
!pip install minio -q

In [2]:
from minio import Minio

client = Minio(
    "minio:9000", 
    access_key="admin",
    secret_key="password123",
    secure=False
)

bucket_name = "minio-test" 

if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Upload_Low_RAM") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memory", "1g") \
    .config("spark.sql.files.maxPartitionBytes", "33554432") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [5]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
bucket_name = "minio-test"


df_postings = spark.read.csv("file:///home/jovyan/work/linkedin_job_postings.csv", header=True, inferSchema=False)
df_postings.repartition(40).write.mode("overwrite").csv(f"s3a://{bucket_name}/raw/linkedin/job_postings/dt={today}/", header=True)

print("Đã xử lý xong file: linkedin_job_postings!")

Đã xử lý xong file: linkedin_job_postings!


In [6]:
# Chạy tiếp 2 file nhỏ sau khi file lớn đã xong xuôi
df_summary = spark.read.csv("file:///home/jovyan/work/job_summary.csv", header=True, inferSchema=False)
df_summary.repartition(20).write.mode("overwrite").csv(f"s3a://{bucket_name}/raw/linkedin/job_summary/dt={today}/", header=True)

df_skills = spark.read.csv("file:///home/jovyan/work/job_skills.csv", header=True, inferSchema=False)
df_skills.repartition(20).write.mode("overwrite").csv(f"s3a://{bucket_name}/raw/linkedin/job_skills/dt={today}/", header=True)

print(" Xong")

 Xong


In [7]:
# Kiểm tra
read_path = f"s3a://{bucket_name}/raw/linkedin/job_postings/dt={today}/"
df_check = spark.read.csv(read_path, header=True)

print(f"Tổng số bản ghi của linkedin_job_postings trên MinIO: {df_check.count():,}")

Tổng số bản ghi của linkedin_job_postings trên MinIO: 1,348,488
